## Data analysis code for gain=5000 data taken on 08/08/2025. Binning 1x1
Used for scanning frequency on solid target. 

In [ ]:
import numpy as np  
import matplotlib.pyplot as plt
import re
import os
from scipy import optimize
from scipy.ndimage import gaussian_filter
import h5py
from scipy.optimize import curve_fit
from scipy.signal import savgol_filter
from scipy.signal import butter, sosfiltfilt, sosfreqz

## Define functions for Gaussian fitting

In [ ]:
def gaussian(p,x):
    return p[0]/(p[1]*(2*np.pi)**.5)*np.exp(-.5*(x-p[2])**2/(p[1]**2))

#p[0]=multiplicative const
#p[1]=std
#p[2]=mean

def residual(p,x,y,dy):
    return (gaussian(p,x)-y)/dy

In [ ]:
def gaussian_fit(counts,probability,probability_uncertainty,p0,plotting_values,show_details):
    ## Implement Gaussian fit
    index_Start, index_End, Plotfit_min, Plotfit_max, Dataplot_min, Dataplot_max = plotting_values
    counts_for_gaussian=counts[index_Start:index_End]
    probability_for_gaussian=probability[index_Start:index_End]
    probability_uncertainty_for_gaussian=probability_uncertainty[index_Start:index_End]
    pf, cov, info, mesg, success = optimize.leastsq(residual, p0, args = (counts_for_gaussian, probability_for_gaussian, 
                                                                          probability_uncertainty_for_gaussian), full_output=1, maxfev=5000)
    ## Plot data and fit
    if cov is None:
        print('fit region: AD count = ', min(counts_for_gaussian), 'to ', max(counts_for_gaussian))
        print('Fit did not converge')
        print('Success code:', success)
        print(mesg)
        fig = plt.figure(figsize=(8,5))
        ax = fig.add_subplot(111)
        ax.errorbar(counts, probability, probability_uncertainty, fmt='k.', label = 'Data',alpha=0.5) 
            #In DarkFrameHistogramFit_polished.ipynb this plots only in the fitted range counts_for_gaussian, but this shouldn't matter if we plot it over all data points.
        x = np.linspace(Plotfit_min,Plotfit_max,1000)
        ax.plot(x, gaussian(p0, x), 'b-', label = 'Guess')
        ax.legend()
        ax.set_title('Gaussian Fit')
        ax.set_xlabel('AD count')
        ax.set_ylabel('Probability')
        ax.set_yscale('log')
        ax.set_xlim([Dataplot_min,Dataplot_max])
        ax.grid()
        plt.show()
    else:
        pferr = [np.sqrt(cov[i,i]) for i in range(len(pf))]
        if show_details:
            print('fit region: AD count = ', min(counts_for_gaussian), 'to ', max(counts_for_gaussian))
            print('Fit Converged')
            chisq = sum(info['fvec']*info['fvec'])
            dof = abs(len(counts_for_gaussian)-len(pf))
            print('Converged with chi-squared', chisq)
            print('Number of degrees of freedom, dof =',dof)
            print('Reduced chi-squared', chisq/dof)
            print('Inital guess values:')
            print('  p0 =', p0)
            print('Best fit values:')
            print('  pf =', list(pf))
            print('Uncertainties in the best fit values:')
            print('  pferr =', pferr)
            print("5 sigma threshold is", pf[2]+pf[1]*5, "with uncertainty", pferr[2]+pferr[1]*5)
            fig = plt.figure(figsize=(8,5))
            ax = fig.add_subplot(111)
            ax.errorbar(counts, probability, probability_uncertainty, fmt='k.', label = 'Data',alpha=0.5)
            x = np.linspace(Plotfit_min,Plotfit_max,1000) #plot fitted curve
            ax.set_yscale('log')
            ax.plot(x, gaussian(pf, x), 'r-', label = 'Fit curve')
            ax.axvspan(min(counts_for_gaussian), max(counts_for_gaussian), alpha=0.4, color='red', label = 'Fitted region')
            ax.set_title('Gaussian Fit', fontsize=16)
            ax.set_xlabel('AD count', fontsize=16)
            ax.set_ylabel('Probability', fontsize=16)
            ax.grid()
            ax.legend(loc=3)
            ax.set_xlim([Dataplot_min,Dataplot_max])
            # ax.set_ylim([1e-6,1e-1])
            plt.tight_layout()
            plt.show()
    return pf

In [ ]:
folder_path_day = r'C:\Experiments\lyman29\BaF_Fluorescence\2025\08\08'

# Photon counting LIF with threshold count=1573

In [ ]:
seq = 102 #sequence number

In [ ]:
count_b_start=1
count_b_end=20

folder_path = folder_path_day + '\\%.4i'%seq
sum_array=[] #total photons
fig = plt.figure(20, figsize=(8,6))
total_counts=0
# get x-cross section for every shot in the sequence
for count_b, filename in enumerate(os.listdir(folder_path)): 
    file_path = os.path.join(folder_path, filename)
    if count_b<=count_b_end and count_b>=count_b_start:
        #if count_b==1 or count_b==2:
        #    print("skip count_b=", count_b)
        #    continue
    # if np.any(np.array([4,6,8,9,12,14,15, 17, 18])==count_b):
        with h5py.File(file_path, 'r') as file:
            image_data = file['images/camera/fluorescence/frame'][:]
            image_data[image_data<1573]=0
            cross_section_x = np.count_nonzero(image_data, axis=0)
            
            # plt.plot(np.arange(512),cross_section_x, label=count_b)
            # sum_array.append(np.sum(cross_section_x))
            plt.plot(np.arange(140,280,1),cross_section_x[140:280], label=count_b)
            sum_array.append(np.sum(cross_section_x[140:280]))
            total_counts+=1


# plt.ylim([1528,1660])
plt.xlabel('x', fontsize=16)
# plt.title('Shots in Seq.7', fontsize=16)
plt.title('Shots in Seq.%i from #%i to #%i' %(seq, count_b_start, count_b_end), fontsize=16)
plt.ylabel('Photon Counts', fontsize=16)
plt.legend()
plt.show()

print("total # of photons = ", np.sum(sum_array))
print("total_counts=", total_counts)
print("average # of photons = ", np.sum(sum_array)/total_counts)


In [ ]:
frequencies=np.arange(348.661109, 348.661329, 0.000020)*1E6 - 35 - 348661174 
print(frequencies)
photons=np.array([12822, 15151, 22935, 22992, 35053, 63060, 125449, 59053, 49176, 55341, 30260, 18562, 13055])
plt.figure(figsize=(8,4))
plt.plot(frequencies, photons, 'bo-')
plt.title("Single Frequency Scan", fontsize=16)
plt.xlabel("Frequency [MHz] Offset from 348661174 MHz", fontsize=16)
plt.ylabel("LIF Photons", fontsize=16)
plt.grid()

# Let's plot

In [ ]:
fields=np.linspace(0, 7.2, num=7)
fields1=np.concatenate((fields, fields[::-1]))
print(fields1)
photons1=np.array([20868, 30493, 32916, 29037, 27264, 27870, 36004, 34583, 36186, 33193, 35972, 33472, 28079, 22851]) #10mW
photons2=np.array([30064, 35476, 25615, 23978, 20155, 19321, 26304, 22599, 18982, 17264, 15991, 14317, 15845, 17534]) #20mW
photons3=np.array([54231, 52371, 53931, 53055, 51933, 50367, 55074, 50000, 49076, 51431, 48581, 44829, 42329, 33693]) #50mW
plt.figure(figsize=(8,4))
plt.plot(fields1, photons3, 'bo-')
plt.title("B-field Scan for 50mW probe power", fontsize=16)
plt.xlabel("B-field Strength", fontsize=16)
plt.ylabel("LIF Photons", fontsize=16)
plt.grid()